In [489]:
import torch
import torch.nn as nn
import torchcvnn

In [490]:
import torch

data = torch.load("data/train_blip_features.pt")

features = data["features"]
questions = data["questions"]
labels = data["labels"]


In [491]:
from torch.utils.data import Dataset

class DisasterVQADataset(Dataset):

    def __init__(self, pt_file):

        data = torch.load(pt_file)

        self.features = data["features"]
        self.questions = data["questions"]
        self.labels = data["labels"]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        return (
            self.features[idx],
            self.questions[idx],
            self.labels[idx]
        )

In [492]:
from torch.utils.data import DataLoader

train_dataset = DisasterVQADataset(
    "data/train_blip_features.pt"
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

features, questions, labels = next(
    iter(train_loader)
)

In [493]:
import pickle

with open("data/word2idx.pkl", "rb") as f:
    word2idx = pickle.load(f)

val_dataset = DisasterVQADataset("data/val_blip_features.pt")

val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

Defining basic functions and channel noise

In [494]:
def real_to_complex(x):

    real = x[..., ::2]
    imag = x[..., 1::2]

    return torch.complex(real, imag)

def complex_awgn(complex_signal, snr_db):

    signal_power = torch.mean(
        torch.abs(complex_signal) ** 2
    )

    snr_linear = 10 ** (snr_db / 10)

    noise_power = signal_power / snr_linear

    noise_std = torch.sqrt(noise_power / 2)

    noise_real = (torch.randn_like(complex_signal.real)* noise_std)

    noise_imag = (
        torch.randn_like(complex_signal.imag)* noise_std)

    noise = torch.complex(noise_real, noise_imag)

    return complex_signal + noise

def complex_to_real(z):

    real = z.real
    imag = z.imag

    return torch.cat([real, imag], dim=-1)

Encoders, decoders, question encoders and classifier

In [495]:
import torch.nn as nn

class SemanticEncoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(768,512),
            nn.ReLU(),

            nn.Linear(512, 256),
            nn.ReLU(),

            nn.Linear(256, 128)
        )


    def forward(self, x):

        return self.encoder(x)

In [496]:
import torchcvnn.nn as c_nn

In [497]:
class ComplexSemanticEncoder(nn.Module):

    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(
            384,
            256
        ).to(torch.cfloat)

        self.act1 = c_nn.Cardioid()

        self.fc2 = nn.Linear(
            256,
            128
        ).to(torch.cfloat)

        self.act2 = c_nn.Cardioid()

        self.fc3 = nn.Linear(
            128,
            64
        ).to(torch.cfloat)

    def forward(self, x):

        x = real_to_complex(x)

        x = self.fc1(x)
        x = self.act1(x)

        x = self.fc2(x)
        x = self.act2(x)

        x = self.fc3(x)

        return x

In [498]:
class SemanticDecoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.decoder = nn.Sequential(

            nn.Linear(128, 256),
            nn.ReLU(),

            nn.Linear(256, 512),
            nn.ReLU(),

            nn.Linear(512, 768)
        )

    def forward(self, x):

        return self.decoder(x)

In [499]:
class ComplexSemanticDecoder(nn.Module):

    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(
            64,
            128
        ).to(torch.cfloat)

        self.act1 = c_nn.Cardioid()

        self.fc2 = nn.Linear(
            128,
            256
        ).to(torch.cfloat)

        self.act2 = c_nn.Cardioid()

        self.fc3 = nn.Linear(
            256,
            384
        ).to(torch.cfloat)

    def forward(self, x):

        x = self.fc1(x)
        x = self.act1(x)

        x = self.fc2(x)
        x = self.act2(x)

        x = self.fc3(x)

        x = complex_to_real(x)

        return x

In [500]:
class QuestionEncoder(nn.Module):

    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=128):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )

    def forward(self, questions):

        embedded = self.embedding(
            questions
        )

        outputs, (hidden, cell) = self.lstm(
            embedded
        )

        question_vector = torch.cat(
            (hidden[-2], hidden[-1]),
            dim=1
        )

        return question_vector

In [501]:
class ComplexQuestionEncoder(nn.Module):

    def __init__(self, vocab_size):

        super().__init__()

        self.base_encoder = QuestionEncoder(vocab_size=vocab_size)

        self.fc1 = nn.Linear(
            128,
            64
        ).to(torch.cfloat)

        self.act1 = c_nn.CReLU()

        self.fc2 = nn.Linear(64, 128).to(torch.cfloat)

    def forward(self, questions):

        x = self.base_encoder(
            questions
        )                   # (B,256)

        x = real_to_complex(
            x.unsqueeze(1)
        ).squeeze(1)        # (B,128) complex

        x = self.fc1(x)

        x = self.act1(x)

        x = self.fc2(x)

        x = complex_to_real(
            x.unsqueeze(1)
        ).squeeze(1)        # (B,256)

        return x

In [502]:
class VQAClassifier(nn.Module):

    def __init__(self):

        super().__init__()

        self.classifier = nn.Sequential(

            nn.Linear(1024, 512),
            nn.ReLU(),

            nn.Linear(512, 128),
            nn.ReLU(),

            nn.Linear(128, 2)
        )

    def forward(self, x):

        return self.classifier(x)

In [503]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [504]:
class CVNNVQA(nn.Module):

    def __init__(self, vocab_size):

        super().__init__()

        self.encoder = ComplexSemanticEncoder()

        self.decoder = ComplexSemanticDecoder()

        self.question_encoder = QuestionEncoder(
            vocab_size=vocab_size
        )

        self.classifier = VQAClassifier()

    def forward(self, image_features, questions):

        compressed = self.encoder(
            image_features
        )

        received = complex_awgn(
            compressed,
            snr_db=5
        )

        reconstructed = self.decoder(
            received
        )

        image_vector = reconstructed.mean(
            dim=1
        )

        question_vector = self.question_encoder(
            questions
        )

        fused = torch.cat(
            [image_vector, question_vector],
            dim=1
        )

        logits = self.classifier(
            fused
        )

        return logits

In [505]:
model = CVNNVQA(
    vocab_size=len(word2idx)
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

C:\Users\Shourya\AppData\Local\Temp\ipykernel_20068\2004815120.py:9: UserWarning: Complex modules are a new feature under active development whose design may change, and some modules might not work as expected when using complex tensors as parameters or buffers. Please file an issue at https://github.com/pytorch/pytorch/issues/new?template=bug-report.yml if a complex module does not work as expected.
  ).to(torch.cfloat)
C:\Users\Shourya\AppData\Local\Temp\ipykernel_20068\2004815120.py:16: UserWarning: Complex modules are a new feature under active development whose design may change, and some modules might not work as expected when using complex tensors as parameters or buffers. Please file an issue at https://github.com/pytorch/pytorch/issues/new?template=bug-report.yml if a complex module does not work as expected.
  ).to(torch.cfloat)
C:\Users\Shourya\AppData\Local\Temp\ipykernel_20068\2004815120.py:23: UserWarning: Complex modules are a new feature under active development whose d

In [506]:
num_epochs = 20

for epoch in range(num_epochs):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for features, questions, labels in train_loader:

        features = features.to(device)
        questions = questions.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(
            features,
            questions
        )

        loss = criterion(
            logits,
            labels
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        preds = logits.argmax(dim=1)

        correct += (
            preds == labels
        ).sum().item()

        total += labels.size(0)

    train_acc = 100 * correct / total

    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"Train Acc: {train_acc:.2f}%"
    )

Epoch 1/20 | Train Acc: 70.21%
Epoch 2/20 | Train Acc: 70.21%
Epoch 3/20 | Train Acc: 71.84%
Epoch 4/20 | Train Acc: 78.98%
Epoch 5/20 | Train Acc: 80.14%
Epoch 6/20 | Train Acc: 81.82%
Epoch 7/20 | Train Acc: 84.32%
Epoch 8/20 | Train Acc: 84.32%
Epoch 9/20 | Train Acc: 85.95%
Epoch 10/20 | Train Acc: 87.40%
Epoch 11/20 | Train Acc: 88.39%
Epoch 12/20 | Train Acc: 89.14%
Epoch 13/20 | Train Acc: 91.11%
Epoch 14/20 | Train Acc: 92.33%
Epoch 15/20 | Train Acc: 92.80%
Epoch 16/20 | Train Acc: 93.61%
Epoch 17/20 | Train Acc: 94.77%
Epoch 18/20 | Train Acc: 95.53%
Epoch 19/20 | Train Acc: 95.53%
Epoch 20/20 | Train Acc: 96.40%


In [507]:
model.eval()

correct = 0
total = 0

all_preds = []
all_labels = []

with torch.no_grad():

    for features, questions, labels in val_loader:

        features = features.to(device)
        questions = questions.to(device)
        labels = labels.to(device)

        logits = model(
            features,
            questions
        )

        preds = logits.argmax(dim=1)

        correct += (
            preds == labels
        ).sum().item()

        total += labels.size(0)

        all_preds.extend(
            preds.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

val_accuracy = 100 * correct / total

print(
    f"Validation Accuracy: {val_accuracy:.2f}%"
)

Validation Accuracy: 78.42%
